# Qwen3-Embedding-0.6B → Qdrant dense snapshot — Kaggle T4 x2

1. Bật **Internet**, chọn **GPU T4 x2**, gắn dataset private `nhminh107/sbtc-datav2-etl-20261016`.
2. Chạy **Save Version → Save & Run All**. Mặc định xử lý **4.342.218 chunk / 1.220.598 document**.
3. Tải toàn bộ Output `qdrant_dense_export`; làm theo `README_RESTORE.md` để restore và kiểm tra dense search.

Mỗi point chỉ chứa named vector **`dense`** từ **Qwen/Qwen3-Embedding-0.6B**, **1024 chiều**, **Cosine**. Không tạo sparse vector/index, không tính hoặc nạp BM25.

Hai process độc lập, mỗi GPU giữ một model chạy **FP16** với SDPA. Last-token pooling với left padding; gộp cửa sổ và chuẩn hóa L2 bằng **float32**, Qdrant lưu vector **float16**. Chunk dài chia cửa sổ **8192 token** rồi gộp theo độ dài, không cắt nội dung. Chunk rỗng dùng một EOS token; không cấp phát ma trận vector toàn corpus trong RAM.

Payload giữ nguyên **`chunk_id`, `parent_id`, `doc_id`, `chunk_text`**, offsets, URL, ngôn ngữ và flags. Hai ID có keyword index trên đĩa; point ID là UUID ổn định từ `chunk_id`. Giữ cả chunk rỗng và `retrieval_eligible=false`.

Vector, payload và HNSW nằm trên đĩa. Bulk ingest tạm `m=0`; khi hai worker thoát mới bật `m=32`. Progress chỉ commit sau upsert `wait=true`. Trước khi manifest ghi `complete=true`, runner restore thật snapshot sang collection tạm và kiểm tra exact count, dense schema/search, keyword indexes và metadata.

Nguồn: [Qwen3](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B), [snapshots](https://qdrant.tech/documentation/operations/snapshots/).


## 1. Thư viện của Kaggle

Cell này chỉ cài vào phiên Kaggle hiện tại, giữ PyTorch/CUDA có sẵn. Không dùng Qdrant local mode vì chế độ đó không tạo được snapshot. Binary Linux musl được ghim phiên bản và kiểm tra SHA-256 trước khi chạy, chỉ lắng nghe localhost; telemetry tắt.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

if not Path('/kaggle/working').is_dir():
    raise RuntimeError('Run this notebook on Kaggle with Internet and GPU T4 x2')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'transformers==4.51.3', 'einops>=0.7', 'psutil>=5.9', 'requests>=2.32',
], check=True)

## 2. Cấu hình và dung lượng

Đọc streaming trực tiếp Input 23,72 GB, không copy corpus vào working. Qdrant state và snapshot tạm nằm trên `/tmp`, phải là ổ đĩa; runner từ chối tmpfs/ramfs. Kiểm tra dung lượng riêng từng filesystem, tính cả payload, HNSW và collection thử restore.

Riêng dense vector float16 1024 chiều có khoảng **8,28 GiB** trước overhead. Snapshot vẫn chứa text, metadata và HNSW, nên chưa bảo đảm vừa quota Output. Giữ ngân sách **18 GiB** để có phần dự phòng. Nếu snapshot chưa vừa thì nén lossless `.snapshot.gz`, xác minh checksum rồi mới copy; chỉ công bố manifest hoàn chỉnh khi export và restore verification thành công.

`resume_from=None` cho run mới. Chỉ resume snapshot dense cùng source hash, model/revision và schema. Collection, state, runner và Output dùng tên riêng; không dùng lại snapshot hybrid hoặc state của notebook trước.


In [ ]:
import torch

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError('Select GPU T4 x2: two CUDA GPUs are required')
for rank in range(2):
    print(f'GPU {rank}: {torch.cuda.get_device_name(rank)}')

DATASET_SLUG = 'sbtc-datav2-etl-20261016'
candidates = [p for p in Path('/kaggle/input').rglob('merged.jsonl') if DATASET_SLUG in str(p)]
if len(candidates) != 1:
    raise FileNotFoundError(f'Attach {DATASET_SLUG}; found {candidates}')
if candidates[0].stat().st_size != 23_724_093_649:
    raise ValueError('Dataset size differs from the uploaded ETL corpus')

runner_dir = Path('/tmp/sbtc_qdrant_dense_runner')
runner_dir.mkdir(parents=True, exist_ok=True)
config = {
    'source': str(candidates[0]),
    'source_sha256': '5b446dab6c1bafb5251441fd18a5ceafcd14837b4e206f4c63c3e80fe9b2adaf',
    'expected_documents': 1_220_598,
    'expected_chunks': 4_342_218,
    'model': 'Qwen/Qwen3-Embedding-0.6B',
    'revision': '97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3',
    'dimension': 1024, 'max_length': 8192,
    'batch_size': 64, 'token_budget': 16_384,
    'buffer_size': 512, 'upsert_batch_size': 128,
    'collection': 'sbtc_qwen3_embedding_0_6b_1024_dense',
    'state_dir': '/tmp/sbtc_qdrant_dense_state',
    'binary': '/tmp/sbtc_qdrant_bin/qdrant',
    'port': 6333, 'qdrant_url': 'http://127.0.0.1:6333',
    'hnsw_m': 32, 'ef_construction': 100,
    'indexing_threshold_kb': 10_000,
    'index_timeout_seconds': 14_400,
    'index_poll_seconds': 15,
    'output': '/kaggle/working/qdrant_dense_export',
    'output_budget_root': '/kaggle/working',
    'output_limit_bytes': 18 * 1024**3,
    'resume_from': None,
}
output = Path(config['output'])
output.mkdir(parents=True, exist_ok=True)
config_path = runner_dir / 'config.json'
config_path.write_text(json.dumps(config, indent=2))
runner_path = runner_dir / 'build_qwen3_dense_qdrant.py'
os.environ['HF_HOME'] = '/tmp/sbtc_qdrant_huggingface'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '30'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Input:', config['source'])
print('State:', config['state_dir'])
print('Output:', config['output'])

## 3. Mã đi kèm notebook

Notebook nhúng runner, restore helper và Python client; không phụ thuộc repository khi upload Kaggle. Output có `qdrant_db.py` để gọi `search_dense(dense_query)` sau restore. Dense query mới dùng cùng Qwen3/revision, last-token pooling, L2 normalization và query instruction trong `README_RESTORE.md`.


In [ ]:
runner_source = r'''"""Kaggle T4 x2: Qwen3-Embedding-0.6B -> verified portable dense-only snapshot."""

import argparse
import gc
import gzip
import hashlib
from itertools import islice
import json
import math
import os
from pathlib import Path
import shutil
import socket
import subprocess
import sys
import tarfile
import time
import uuid
import zlib

import numpy as np
import psutil
import requests

QDRANT_VERSION = "1.19.2"
QDRANT_URL = f"https://github.com/qdrant/qdrant/releases/download/v{QDRANT_VERSION}/qdrant-x86_64-unknown-linux-musl.tar.gz"
QDRANT_SHA256 = "50b253243309ed0ae50a19f678f0d0adcc4f0b567b3bd2c5c0c7e02fa8404bb6"
DENSE_VECTOR = "dense"


def vector_schema(config: dict) -> dict:
    return {"dense": DENSE_VECTOR, "dense_datatype": "float16"}





def padded_batches(order: list[int], lengths: list[int], batch_size: int, token_budget: int):
    offset = 0
    while offset < len(order):
        longest = lengths[order[offset]]
        size = min(batch_size, max(1, token_budget // longest))
        yield order[offset:offset + size]
        offset += size

def normalize_vectors(vectors: np.ndarray) -> np.ndarray:
    vectors = np.ascontiguousarray(vectors, dtype=np.float32)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    if not np.isfinite(vectors).all() or np.any(norms <= 1e-12):
        raise ValueError("Non-finite or zero embedding; no vector was inserted")
    return vectors / norms

def encode_records(records: list[dict], tokenizer, forward, config: dict) -> tuple[np.ndarray, int]:
    tokens = tokenizer(
        [record["text"] for record in records], add_special_tokens=False,
        truncation=False, return_attention_mask=False, return_token_type_ids=False,
    )["input_ids"]
    # Qwen does not add BOS/EOS automatically to document text.
    payload = config["max_length"]
    windows, owners, weights = [], [], []
    long_chunks = 0
    for owner, ids in enumerate(tokens):
        long_chunks += len(ids) > payload
        for start in range(0, max(1, len(ids)), payload):
            piece = ids[start:start + payload]
            # Qwen adds no special tokens; keep empty chunks as one EOS token.
            if not piece:
                piece = [tokenizer.eos_token_id]
            windows.append({"input_ids": piece})
            owners.append(owner)
            weights.append(max(1, len(piece)))
    lengths = [len(window["input_ids"]) for window in windows]
    order = sorted(range(len(windows)), key=lengths.__getitem__, reverse=True)
    sums = np.zeros((len(records), config["dimension"]), dtype=np.float32)
    for positions in padded_batches(order, lengths, config["batch_size"], config["token_budget"]):
        vectors = forward([windows[position] for position in positions])
        for position, vector in zip(positions, vectors, strict=True):
            sums[owners[position]] += vector * weights[position]
    return normalize_vectors(sums), long_chunks

def load_encoder(config: dict):
    import torch
    from transformers import AutoModel, AutoTokenizer

    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
        raise RuntimeError("Each worker must see exactly one usable CUDA GPU")
    torch.set_num_threads(1)
    device = torch.device("cuda:0")
    torch.cuda.synchronize(device)
    tokenizer = AutoTokenizer.from_pretrained(
        config["model"], revision=config["revision"], padding_side="left",
    )
    model = AutoModel.from_pretrained(
        config["model"], revision=config["revision"], attn_implementation="sdpa",
        torch_dtype=torch.float16, low_cpu_mem_usage=False,
    )
    model.config.use_cache = False
    if model.config.hidden_size != config["dimension"]:
        raise ValueError("Unexpected model embedding dimension")
    model.eval().to(device)
    print(f"GPU {config['rank']}: {torch.cuda.get_device_name(0)}; SDPA; float16", flush=True)

    def forward(windows: list[dict]) -> np.ndarray:
        def infer(items: list[dict]) -> np.ndarray:
            features = tokenizer.pad(items, padding=True, return_tensors="pt")
            features = {name: tensor.to(device) for name, tensor in features.items()}
            with torch.inference_mode():
                # Left padding keeps the last real token at the final position.
                embeddings = model(**features, use_cache=False).last_hidden_state[:, -1].float()
            return normalize_vectors(embeddings.cpu().numpy())

        retry = False
        try:
            return infer(windows)
        except torch.cuda.OutOfMemoryError:
            if len(windows) == 1:
                raise RuntimeError("CUDA OOM at batch size 1; no chunk was skipped") from None
            retry = True
        if retry:
            # Release exception frames before retrying smaller batches.
            gc.collect()
            torch.cuda.empty_cache()
            middle = len(windows) // 2
            config["batch_size"] = min(config["batch_size"], middle)
            print(f"GPU {config['rank']}: OOM, retry batch_size={middle}", flush=True)
            return np.concatenate([forward(windows[:middle]), forward(windows[middle:])])

    return tokenizer, forward


def write_json(path: Path, value: dict) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2)
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)


def checksum(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()


def signature(config: dict) -> dict:
    return {**{key: config[key] for key in (
        "source_sha256", "expected_documents", "expected_chunks", "model", "revision",
        "dimension", "max_length", "collection",
    )}, "vector_schema": vector_schema(config), "payload_schema_version": 2}


def make_record(document: dict, child: dict, line: int) -> dict:
    text = child["chunk_text"]
    if not isinstance(text, str):
        raise ValueError(f"Non-string chunk_text at source line {line}")
    for field in ("chunk_id", "parent_id"):
        if not isinstance(child.get(field), str) or not child[field].strip():
            raise ValueError(f"Missing or invalid {field} at source line {line}")
    if "doc_id" in child and child["doc_id"] != document["doc_id"]:
        raise ValueError(f"Child doc_id differs from its document at source line {line}")
    return {
        "chunk_id": child["chunk_id"], "doc_id": document["doc_id"],
        "parent_id": child["parent_id"], "chunk_order": child["chunk_order"],
        "start_char": child["start_char"], "end_char": child["end_char"],
        "source_line": line, "url": document.get("url"),
        "source_language": document.get("source_language"),
        "retrieval_eligible": child.get("retrieval_eligible"),
        "empty_text": not text.strip(), "chunk_text": text,
    }


def iter_records(source: Path, rank: int):
    with source.open(encoding="utf-8") as stream:
        for line, raw in enumerate(stream, start=1):
            if (line - 1) % 2 != rank:
                continue
            document = json.loads(raw)
            for child in document.get("children", []):
                yield make_record(document, child, line)


def validate_source(config: dict) -> dict:
    digest, compressor = hashlib.sha256(), zlib.compressobj(level=1, wbits=31)
    documents = payload_bytes = compressed_bytes = 0
    shards = [0, 0]
    with Path(config["source"]).open("rb") as stream:
        for documents, raw in enumerate(stream, start=1):
            digest.update(raw)
            document = json.loads(raw)
            parent_ids = {parent["parent_id"] for parent in document.get("parents", [])}
            for child in document.get("children", []):
                record = make_record(document, child, documents)
                if record["parent_id"] not in parent_ids:
                    raise ValueError(f"Unknown parent_id at source line {documents}: {record['parent_id']}")
                payload = (json.dumps(record, ensure_ascii=False) + "\n").encode("utf-8")
                payload_bytes += len(payload)
                compressed_bytes += len(compressor.compress(payload))
                shards[(documents - 1) % 2] += 1
            if documents % 200_000 == 0:
                print(f"Validate: {documents:,} documents; {sum(shards):,} chunks", flush=True)
    compressed_bytes += len(compressor.flush())
    if digest.hexdigest() != config["source_sha256"]:
        raise ValueError("Source SHA-256 differs from the uploaded ETL corpus")
    if documents != config["expected_documents"] or sum(shards) != config["expected_chunks"]:
        raise ValueError(f"Unexpected corpus counts: {documents} documents; {sum(shards)} chunks")
    return {"documents": documents, "chunks": sum(shards), "shards": shards,
            "sha256": digest.hexdigest(), "payload_bytes": payload_bytes,
            "gzip_payload_bytes": compressed_bytes}


def output_budget(config: dict) -> int:
    root = Path(config["output_budget_root"])
    used = sum(path.stat().st_size for path in root.rglob("*") if path.is_file())
    return max(0, min(config["output_limit_bytes"] - used,
                      shutil.disk_usage(root).free - 512 * 1024**2))


def check_resources(config: dict, source: dict) -> None:
    state = Path(config["state_dir"])
    mounts = [mount for mount in psutil.disk_partitions(all=True)
              if state.resolve().is_relative_to(Path(mount.mountpoint))]
    kind = max(mounts, key=lambda mount: len(mount.mountpoint)).fstype if mounts else "unknown"
    if kind in {"tmpfs", "ramfs"}:
        raise RuntimeError(f"Qdrant state must be on disk, not {kind}: {state}")
    vectors = config["expected_chunks"] * config["dimension"] * 2
    graph = config["expected_chunks"] * (8 * config["hnsw_m"] + 128)
    collection = vectors + graph + source["payload_bytes"]
    # Covers segment rewrites, the collection, snapshot temporary data and export.
    required = 4 * collection + 4 * 1024**3
    free = shutil.disk_usage(state).free
    print(f"Qdrant disk ({kind}): free={free / 1024**3:.2f} GiB; budget={required / 1024**3:.2f} GiB", flush=True)
    if free < required:
        raise RuntimeError("Not enough state disk space for Qdrant plus snapshot copies")
    estimated_export = vectors + graph + int(source["gzip_payload_bytes"] * 2) + 512 * 1024**2
    budget = output_budget(config)
    print(f"Output: remaining={budget / 1024**3:.2f} GiB; estimated compressed snapshot={estimated_export / 1024**3:.2f} GiB", flush=True)
    if budget < estimated_export:
        print("WARNING: dense export may exceed Output quota; final size is checked exactly. "
              "An oversized snapshot stays in state and the run is not marked complete.", flush=True)
    if budget < vectors * 0.8 + 512 * 1024**2:
        raise RuntimeError("Output budget cannot reasonably hold the dense vectors alone")
    host = psutil.virtual_memory()
    maximum = Path("/sys/fs/cgroup/memory.max")
    limit = int(maximum.read_text()) if maximum.exists() and maximum.read_text().strip().isdigit() else host.total
    print(f"RAM: host={host.total / 1024**3:.1f} GiB; cgroup limit={limit / 1024**3:.1f} GiB; disk-backed vectors/payload/HNSW", flush=True)
    if min(host.total, limit) < 8 * 1024**3:
        raise RuntimeError("Two model workers plus Qdrant require at least 8 GiB RAM")


def download_qdrant(config: dict) -> Path:
    binary = Path(config["binary"])
    if binary.exists():
        return binary
    binary.parent.mkdir(parents=True, exist_ok=True)
    archive = binary.with_suffix(".tar.gz")
    with requests.get(QDRANT_URL, stream=True, timeout=(30, 120)) as response:
        response.raise_for_status()
        with archive.open("wb") as stream:
            for block in response.iter_content(1024**2):
                stream.write(block)
    if checksum(archive) != QDRANT_SHA256:
        raise ValueError("Official Qdrant release archive SHA-256 mismatch")
    with tarfile.open(archive) as bundle:
        member = next(item for item in bundle.getmembers()
                      if Path(item.name).name == "qdrant" and item.isfile())
        with bundle.extractfile(member) as source, binary.open("wb") as destination:
            shutil.copyfileobj(source, destination)
    binary.chmod(0o755)
    return binary


def api(config: dict, method: str, path: str, body=None, session=None, timeout=300):
    connection = session or requests
    response = connection.request(method, config["qdrant_url"] + path, json=body, timeout=timeout)
    if not response.ok:
        raise RuntimeError(f"Qdrant {method} {path}: HTTP {response.status_code}: {response.text[:1500]}")
    return response.json().get("result")


def start_server(config: dict):
    binary = download_qdrant(config)
    port = config["port"]
    with socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", port)) == 0:
            raise RuntimeError(f"Port {port} is already in use; refusing to attach to another database")
    state = Path(config["state_dir"])
    for directory in ("storage", "snapshots", "snapshot_temp"):
        (state / directory).mkdir(parents=True, exist_ok=True)
    server_config = {
        "log_level": "WARN", "telemetry_disabled": True,
        "storage": {"storage_path": str(state / "storage"),
                    "snapshots_path": str(state / "snapshots"),
                    "temp_path": str(state / "snapshot_temp"),
                    "on_disk_payload": True, "low_memory_mode": "no_populate",
                    "performance": {"max_search_threads": 2, "optimizer_cpu_budget": 2}},
        "service": {"host": "127.0.0.1", "http_port": port, "grpc_port": port + 1,
                    "max_request_size_mb": 64, "max_workers": 2},
    }
    # JSON syntax is valid YAML; Qdrant's loader expects a .yaml config path.
    config_path = state / "qdrant_config.yaml"
    write_json(config_path, server_config)
    log = (Path(config["output"]) / "qdrant.log").open("a", encoding="utf-8")
    process = subprocess.Popen([str(binary), "--config-path", str(config_path), "--disable-telemetry"],
                               cwd=state, stdout=log, stderr=subprocess.STDOUT)
    try:
        for _ in range(120):
            if process.poll() is not None:
                raise RuntimeError("Qdrant exited during startup; inspect qdrant.log")
            try:
                response = requests.get(config["qdrant_url"] + "/readyz", timeout=1)
                if response.ok:
                    version = requests.get(config["qdrant_url"], timeout=5).json()["version"]
                    if version != QDRANT_VERSION:
                        raise RuntimeError(f"Expected Qdrant {QDRANT_VERSION}; server reports {version}")
                    print(f"Qdrant {version} READY at {config['qdrant_url']}", flush=True)
                    return process, log
            except requests.ConnectionError:
                pass
            time.sleep(0.5)
        raise TimeoutError("Qdrant did not become ready in 60 seconds")
    except BaseException:
        stop_server(process, log)
        raise


def stop_server(process, log) -> None:
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    log.close()


def create_collection(config: dict) -> None:
    name = config["collection"]
    response = requests.get(config["qdrant_url"] + f"/collections/{name}", timeout=30)
    if response.status_code != 404:
        response.raise_for_status()
        params = response.json()["result"]["config"]["params"]
        vectors = params["vectors"].get(DENSE_VECTOR, {})
        if (vectors.get("size") != config["dimension"] or vectors.get("distance") != "Cosine"
                or vectors.get("datatype") != "float16" or params.get("sparse_vectors")):
            raise ValueError("Existing collection is not the matching dense-only schema; use new state")
    else:
        api(config, "PUT", f"/collections/{name}", {
            "vectors": {DENSE_VECTOR: {"size": config["dimension"], "distance": "Cosine", "on_disk": True,
                                       "datatype": "float16"}},
            "on_disk_payload": True, "shard_number": 1,
            "hnsw_config": {"m": 0, "on_disk": True, "max_indexing_threads": 2},
            "optimizers_config": {"default_segment_number": 2, "max_segment_size": 500_000,
                                  "indexing_threshold": config["indexing_threshold_kb"],
                                  "max_optimization_threads": 1},
        })
    for field in ("chunk_id", "parent_id"):
        api(config, "PUT", f"/collections/{name}/index?wait=true",
            {"field_name": field, "field_schema": {"type": "keyword", "on_disk": True}})


def point_id(chunk_id: str) -> str:
    return str(uuid.uuid5(uuid.NAMESPACE_URL, "sbtc:" + chunk_id))


def upsert_batch(config: dict, records: list[dict], vectors: np.ndarray, session) -> None:
    for start in range(0, len(records), config["upsert_batch_size"]):
        rows = records[start:start + config["upsert_batch_size"]]
        points = [{"id": point_id(row["chunk_id"]),
                   "vector": {DENSE_VECTOR: vector.tolist()},
                   "payload": row}
                  for row, vector in zip(rows, vectors[start:start + len(rows)], strict=True)]
        # An acknowledgement can be lost after Qdrant commits. Stable UUIDs make
        # replaying this upsert safe; progress is written only after wait=true.
        for attempt in range(3):
            try:
                result = api(config, "PUT", f"/collections/{config['collection']}/points?wait=true",
                             {"points": points}, session=session)
                if result["status"] != "completed":
                    raise RuntimeError(f"Upsert was not completed: {result}")
                break
            except (requests.ConnectionError, requests.Timeout):
                if attempt == 2:
                    raise
                time.sleep(2**attempt)


def progress_path(config: dict, rank: int) -> Path:
    return Path(config["output"]) / f"gpu{rank}.progress.json"


def run_worker(config: dict, rank: int) -> None:
    config = {**config, "rank": rank}
    progress = progress_path(config, rank)
    saved = json.loads(progress.read_text()) if progress.exists() else {"chunks": 0, "long_chunks": 0}
    count, long_chunks = saved["chunks"], saved["long_chunks"]
    expected = config["shards"][rank]
    if count == expected:
        print(f"GPU {rank}: shard already complete ({count:,} chunks)", flush=True)
        return
    if count > expected:
        raise ValueError("Checkpoint count exceeds source shard size")
    tokenizer, forward = load_encoder(config)
    smoke = [{"text": text} for text in ("English test.", "Kiểm tra tiếng Việt.", "中文测试。")]
    encode_records(smoke, tokenizer, forward, config)
    started, resumed = time.monotonic(), count
    records = []
    with requests.Session() as session:
        def commit():
            nonlocal count, long_chunks
            vectors, long_count = encode_records(
                [{"text": row["chunk_text"]} for row in records], tokenizer, forward, config,
            )
            upsert_batch(config, records, vectors, session)
            count += len(records)
            long_chunks += long_count
            write_json(progress, {"chunks": count, "long_chunks": long_chunks})
            if count % (config["buffer_size"] * 20) == 0 or count == expected:
                speed = (count - resumed) / max(0.001, time.monotonic() - started)
                print(f"GPU {rank}: {count:,}/{expected:,} chunks committed to Qdrant; {speed:.1f} chunks/s", flush=True)
            records.clear()

        for record in islice(iter_records(Path(config["source"]), rank), count, None):
            records.append(record)
            if len(records) == config["buffer_size"]:
                commit()
        if records:
            commit()
    if count != expected:
        raise ValueError(f"GPU {rank} count mismatch: {count} != {expected}")


def embed_two_gpus(config: dict, config_path: Path) -> None:
    import torch
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        raise RuntimeError("Select GPU T4 x2: two CUDA GPUs are required")
    workers = []
    try:
        for rank in range(2):
            environment = {**os.environ, "CUDA_VISIBLE_DEVICES": str(rank),
                           "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1",
                           "TOKENIZERS_PARALLELISM": "false"}
            workers.append(subprocess.Popen([
                sys.executable, "-u", __file__, "worker", "--config", str(config_path), "--rank", str(rank),
            ], env=environment))
        while any(worker.poll() is None for worker in workers):
            if any(worker.poll() not in (None, 0) for worker in workers):
                raise RuntimeError("An embedding worker failed; stopping its peer before recovery snapshot")
            time.sleep(1)
        if any(worker.returncode for worker in workers):
            raise RuntimeError(f"Embedding workers failed: {[worker.returncode for worker in workers]}")
    finally:
        for worker in workers:
            if worker.poll() is None:
                worker.terminate()
        for worker in workers:
            try:
                worker.wait(timeout=30)
            except subprocess.TimeoutExpired:
                worker.kill()
                worker.wait()


def count_points(config: dict) -> int:
    return api(config, "POST", f"/collections/{config['collection']}/points/count", {"exact": True})["count"]


def wait_for_index(config: dict) -> dict:
    name = config["collection"]
    api(config, "PATCH", f"/collections/{name}", {
        "hnsw_config": {"m": config["hnsw_m"], "ef_construct": config["ef_construction"],
                        "on_disk": True, "max_indexing_threads": 2},
    })
    deadline = time.monotonic() + config["index_timeout_seconds"]
    stable = 0
    while time.monotonic() < deadline:
        info = api(config, "GET", f"/collections/{name}")
        if info["optimizer_status"] != "ok" or info["status"] == "red":
            raise RuntimeError(f"Qdrant optimizer error: {info['optimizer_status']}")
        # Small appendable segments below indexing_threshold are intentionally
        # left as plain vectors. Do not wait for indexed_vectors_count == total.
        allowance = info["segments_count"] * math.ceil(
            config["indexing_threshold_kb"] * 1024 / (config["dimension"] * 2),
        )
        indexed = info.get("indexed_vectors_count", 0)
        ready = info["status"] == "green" and indexed >= config["expected_chunks"] - allowance
        stable = stable + 1 if ready else 0
        print(f"Index: status={info['status']}; indexed={indexed:,}; points={info['points_count']:,}; segments={info['segments_count']}", flush=True)
        if stable >= 3:
            return info
        time.sleep(config.get("index_poll_seconds", 15))
    raise TimeoutError("HNSW optimization timed out; export a recovery snapshot instead of claiming success")


def check_search(config: dict) -> dict:
    name = config["collection"]
    info = api(config, "GET", f"/collections/{name}")
    params = info["config"]["params"]
    dense = params["vectors"].get(DENSE_VECTOR, {})
    if (dense.get("size") != config["dimension"] or dense.get("distance") != "Cosine"
            or dense.get("datatype") != "float16" or params.get("sparse_vectors")):
        raise ValueError("Stored collection is not the expected float16 dense-only schema")
    points = api(config, "POST", f"/collections/{name}/points/scroll",
                 {"limit": 64, "with_vector": True, "with_payload": True,
                  "filter": {"must": [{"key": "empty_text", "match": {"value": False}}]}})["points"]
    if not points:
        raise ValueError("Collection has no nonempty text for dense verification")
    point = points[0]
    payload, vectors = point["payload"], point["vector"]
    if not all(payload.get(key) for key in ("chunk_text", "chunk_id", "parent_id")):
        raise ValueError("Missing chunk_text/chunk_id/parent_id in restored payload")
    if point["id"] != point_id(payload["chunk_id"]) or len(vectors[DENSE_VECTOR]) != config["dimension"]:
        raise ValueError("Stored point ID or dense vector does not match the source contract")
    result = api(config, "POST", f"/collections/{name}/points/query", {
        "query": vectors[DENSE_VECTOR], "using": DENSE_VECTOR, "limit": 3, "with_payload": True,
    })["points"]
    if not result or result[0]["score"] < 0.99:
        raise ValueError("Stored-vector similarity smoke test failed")
    for hit in result:
        if not all(key in hit["payload"] for key in ("chunk_id", "parent_id", "chunk_text")):
            raise ValueError("Dense results lost source metadata")
    filtered = api(config, "POST", f"/collections/{name}/points/scroll", {
        "limit": 1, "with_payload": True,
        "filter": {"must": [{"key": "chunk_id", "match": {"value": payload["chunk_id"]}},
                             {"key": "parent_id", "match": {"value": payload["parent_id"]}}]},
    })["points"]
    if not filtered or filtered[0]["payload"] != payload:
        raise ValueError("chunk_id/parent_id payload lookup did not round-trip")
    print("SEARCH PASSED: dense and chunk/parent metadata", flush=True)
    return {"point_id": point["id"], "chunk_id": payload["chunk_id"], "parent_id": payload["parent_id"],
            "dense": True, "payload_lookup": True}


def verify_snapshot_restore(config: dict, raw: Path, points: int) -> dict:
    name = config["collection"] + "_restore_check"
    existing = api(config, "GET", "/collections")["collections"]
    if any(item["name"] == name for item in existing):
        raise ValueError(f"Restore verification collection already exists: {name}")
    api(config, "PUT", f"/collections/{name}/snapshots/recover?wait=true",
        {"location": raw.resolve().as_uri(), "priority": "snapshot"}, timeout=3600)
    try:
        restored = {**config, "collection": name}
        if count_points(restored) != points:
            raise ValueError("Snapshot restore changed the exact point count")
        info = api(restored, "GET", f"/collections/{name}")
        for field in ("chunk_id", "parent_id"):
            if field not in info["payload_schema"]:
                raise ValueError(f"Restored snapshot lost the {field} keyword index")
        report = check_search(restored)
        return {"verified": True, "points": points, "search": report}
    finally:
        # Only the temporary collection created above belongs to this check.
        api(config, "DELETE", f"/collections/{name}?timeout=120")


def export_snapshot(config: dict, complete: bool) -> dict:
    name = config["collection"]
    # A terminated worker can leave a previously accepted upsert on the server.
    # Record a count only when it stays stable across snapshot creation.
    for attempt in range(2):
        before = count_points(config)
        result = api(config, "POST", f"/collections/{name}/snapshots?wait=true", timeout=3600)
        points = count_points(config)
        if points == before:
            break
        if attempt == 1:
            raise RuntimeError("Collection kept changing during snapshot creation; manifest was not published")
    raw = Path(config["state_dir"]) / "snapshots" / name / Path(result["name"]).name
    if not raw.is_file() or not tarfile.is_tarfile(raw):
        raise ValueError("Qdrant did not produce a valid collection snapshot archive")
    raw_sha256 = checksum(raw)
    verification = verify_snapshot_restore(config, raw, points) if complete else {"verified": False}
    prefix = "sbtc_qwen3_embedding_0_6b_1024_dense" if complete else "RECOVERY_sbtc_qwen3_embedding_0_6b_1024_dense"
    source = raw
    if raw.stat().st_size > output_budget(config):
        source = Path(config["state_dir"]) / f"{prefix}.snapshot.gz"
        print("Compressing snapshot losslessly outside Kaggle Output", flush=True)
        with raw.open("rb") as reader, gzip.open(source, "wb", compresslevel=6) as writer:
            shutil.copyfileobj(reader, writer, length=8 * 1024**2)
        digest = hashlib.sha256()
        with gzip.open(source, "rb") as reader:
            for block in iter(lambda: reader.read(8 * 1024**2), b""):
                digest.update(block)
        if digest.hexdigest() != raw_sha256:
            raise ValueError("Compressed snapshot does not match the original snapshot")
    if source.stat().st_size > output_budget(config):
        raise RuntimeError(f"Snapshot exceeds Output quota even after compression. Original kept at {raw}")
    suffix = ".snapshot.gz" if source != raw else ".snapshot"
    target = Path(config["output"]) / (prefix + suffix)
    if target.exists():
        raise FileExistsError(f"Snapshot output exists: {target}")
    partial = target.with_suffix(target.suffix + ".partial")
    shutil.copyfile(source, partial)
    exported_sha256 = checksum(partial)
    if exported_sha256 != checksum(source):
        raise ValueError("Exported snapshot checksum mismatch")
    partial.replace(target)
    progress = {}
    for rank in range(2):
        path = progress_path(config, rank)
        progress[str(rank)] = json.loads(path.read_text()) if path.exists() else {"chunks": 0, "long_chunks": 0}
    manifest = {
        "complete": complete, "collection": name, "qdrant_version": QDRANT_VERSION,
        "signature": signature(config), "points": points, "progress": progress,
        "vector_schema": vector_schema(config), "restore_verification": verification,
        "snapshot_file": target.name, "snapshot_bytes": target.stat().st_size,
        "snapshot_sha256": exported_sha256, "raw_snapshot_sha256": raw_sha256,
        "raw_snapshot_bytes": raw.stat().st_size,
        "embedding": {"pooling": "last_token", "normalized": True, "inference": "float16",
                      "storage": "float16", "long_text": "8192-token windows, content-length-weighted mean, L2"},
        "payload": "chunk_text, chunk_id, doc_id, parent_id, offsets, URL, language and flags",
        "payload_indexes": {"chunk_id": "keyword", "parent_id": "keyword"},
    }
    write_json(Path(config["output"]) / "snapshot_manifest.json", manifest)
    print(f"{'SUCCESS' if complete else 'RECOVERY ONLY'}: {target}; {manifest['points']:,} points", flush=True)
    return manifest


def restore_for_resume(config: dict) -> None:
    folder = Path(config["resume_from"])
    manifest = json.loads((folder / "snapshot_manifest.json").read_text())
    if manifest["signature"] != signature(config) or manifest["qdrant_version"] != QDRANT_VERSION:
        raise ValueError("Resume snapshot source/model/Qdrant version does not match")
    snapshot = folder / manifest["snapshot_file"]
    if checksum(snapshot) != manifest["snapshot_sha256"]:
        raise ValueError("Resume snapshot checksum mismatch")
    # file:// recovery is restricted to the server's snapshots directory.
    raw = Path(config["state_dir"]) / "snapshots" / "resume.snapshot"
    opener = gzip.open if snapshot.suffix == ".gz" else open
    with opener(snapshot, "rb") as source, raw.open("wb") as destination:
        shutil.copyfileobj(source, destination, length=8 * 1024**2)
    if checksum(raw) != manifest["raw_snapshot_sha256"]:
        raise ValueError("Unpacked resume snapshot checksum mismatch")
    api(config, "PUT", f"/collections/{config['collection']}/snapshots/recover?wait=true", {
        "location": raw.resolve().as_uri(), "priority": "snapshot",
    }, timeout=3600)
    if count_points(config) != manifest["points"]:
        raise ValueError("Restored snapshot point count differs from its manifest")
    for rank in range(2):
        write_json(progress_path(config, rank), manifest["progress"][str(rank)])
    print(f"Restored {manifest['points']:,} points from attached snapshot", flush=True)


def run_pipeline(config: dict, config_path: Path) -> None:
    output, state = Path(config["output"]), Path(config["state_dir"])
    for path in (output, state, Path(config["output_budget_root"])):
        path.mkdir(parents=True, exist_ok=True)
    if (output / "snapshot_manifest.json").exists():
        raise FileExistsError("Snapshot output exists; choose a new output directory")
    saved = state / "run_signature.json"
    if saved.exists() and json.loads(saved.read_text()) != signature(config):
        raise ValueError("State directory belongs to a different corpus/model; choose a new state directory")
    write_json(saved, signature(config))
    source = validate_source(config)
    write_json(output / "source_manifest.json", source)
    check_resources(config, source)
    config["shards"] = source["shards"]
    write_json(config_path, config)
    process = log = None
    collection_ready = False
    try:
        process, log = start_server(config)
        if config.get("resume_from"):
            # Restore is permitted only into an empty server owned by this run.
            if api(config, "GET", "/collections")["collections"]:
                raise ValueError("Resume requires a new empty Qdrant state directory")
            restore_for_resume(config)
        create_collection(config)
        collection_ready = True
        committed = sum(json.loads(progress_path(config, rank).read_text())["chunks"]
                        for rank in range(2) if progress_path(config, rank).exists())
        if count_points(config) < committed:
            raise ValueError("Progress files exist but Qdrant data is missing; restore the matching snapshot")
        # Snapshot/restore support is tested before the costly GPU phase by the
        # companion integration test; real server readiness is checked here.
        print("PREFLIGHT PASSED: source, storage and real Qdrant server verified", flush=True)
        embed_two_gpus(config, config_path)
        if count_points(config) != config["expected_chunks"]:
            raise ValueError("Qdrant count differs from the corpus (missing or duplicate chunk IDs)")
        wait_for_index(config)
        search_report = check_search(config)
        sample = api(config, "POST", f"/collections/{config['collection']}/points", {
            "ids": [search_report["point_id"]], "with_payload": True, "with_vector": True,
        })[0]
        write_json(output / "dense_query_example.json", {
            "text": sample["payload"]["chunk_text"], "dense_vector": sample["vector"][DENSE_VECTOR],
            "chunk_id": sample["payload"]["chunk_id"], "parent_id": sample["payload"]["parent_id"],
            "note": "Stored-vector restore smoke test. Encode each new query with the pinned Qwen3 model with last-token pooling and the query instruction in README_RESTORE.md.",
        })
        export_snapshot(config, complete=True)
        write_json(output / "pipeline_status.json", {"success": True, "stage": "snapshot_exported"})
    except Exception as error:
        write_json(output / "pipeline_status.json", {"success": False, "error": str(error)})
        print(f"PIPELINE FAILED: {error}", flush=True)
        if collection_ready:
            try:
                if count_points(config):
                    export_snapshot(config, complete=False)
            except Exception as recovery_error:
                print(f"Recovery snapshot failed: {recovery_error}. Database remains at {state}", flush=True)
        raise
    finally:
        if process is not None:
            stop_server(process, log)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("stage", choices=["run", "worker"])
    parser.add_argument("--config", type=Path, required=True)
    parser.add_argument("--rank", type=int, choices=[0, 1])
    args = parser.parse_args()
    config = json.loads(args.config.read_text())
    if args.stage == "worker":
        run_worker(config, args.rank)
    else:
        run_pipeline(config, args.config)


if __name__ == "__main__":
    main()
'''

restore_source = r'''"""Verify and unpack a notebook export; print commands to restore with Docker."""

import argparse
import gzip
import hashlib
import json
from pathlib import Path
import shlex
import shutil
from urllib.parse import quote
from urllib.request import Request, urlopen


def verify(folder: Path, url: str) -> dict:
    manifest = json.loads((folder / "snapshot_manifest.json").read_text())
    if not manifest["complete"]:
        raise ValueError("Recovery snapshot is incomplete; resume the notebook before final verification")
    name = quote(manifest["collection"], safe="")

    def request(method, path, body=None):
        data = json.dumps(body).encode() if body is not None else None
        req = Request(url.rstrip("/") + path, data=data, method=method,
                      headers={"Content-Type": "application/json"})
        with urlopen(req, timeout=300) as response:
            return json.load(response)["result"]

    count = request("POST", f"/collections/{name}/points/count", {"exact": True})["count"]
    if count != manifest["points"]:
        raise ValueError("Restored exact point count differs from manifest")
    schema = manifest["vector_schema"]
    info = request("GET", f"/collections/{name}")
    params = info["config"]["params"]
    dense = params["vectors"].get(schema["dense"], {})
    if (dense.get("size") != manifest["signature"]["dimension"] or dense.get("distance") != "Cosine"
            or dense.get("datatype") != schema["dense_datatype"] or params.get("sparse_vectors")):
        raise ValueError("Restored collection is not the expected float16 dense-only schema")
    for field in ("chunk_id", "parent_id"):
        if field not in info["payload_schema"]:
            raise ValueError(f"Missing restored {field} keyword index")
    points = request("POST", f"/collections/{name}/points/scroll", {
        "limit": 64, "with_vector": True, "with_payload": True,
        "filter": {"must": [{"key": "empty_text", "match": {"value": False}}]},
    })["points"]
    if not points:
        raise ValueError("No dense sample found after restore")
    point = points[0]
    payload = point["payload"]
    if not all(payload.get(key) for key in ("chunk_text", "chunk_id", "parent_id")):
        raise ValueError("Restored sample lost text/chunk/parent metadata")
    vector = point["vector"][schema["dense"]]
    if len(vector) != dense["size"]:
        raise ValueError("Restored dense vector dimension differs from manifest")
    hits = request("POST", f"/collections/{name}/points/query", {
        "query": vector, "using": schema["dense"], "limit": 3, "with_payload": True,
    })["points"]
    if not hits or hits[0]["score"] < 0.99:
        raise ValueError("Restored dense search failed")
    if any(not all(key in hit["payload"] for key in ("chunk_id", "parent_id", "chunk_text")) for hit in hits):
        raise ValueError("Dense results lost text/chunk/parent metadata")
    filtered = request("POST", f"/collections/{name}/points/scroll", {
        "limit": 1, "with_payload": True,
        "filter": {"must": [{"key": "chunk_id", "match": {"value": payload["chunk_id"]}},
                             {"key": "parent_id", "match": {"value": payload["parent_id"]}}]},
    })["points"]
    if not filtered or filtered[0]["payload"] != payload:
        raise ValueError("Restored chunk_id/parent_id lookup did not round-trip")
    return {"points": count, "dense": True, "payload_lookup": True,
            "sample_chunk_id": payload["chunk_id"], "sample_parent_id": payload["parent_id"]}


def checksum(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()


def unpack(folder: Path) -> tuple[dict, Path]:
    manifest = json.loads((folder / "snapshot_manifest.json").read_text())
    snapshot = folder / manifest["snapshot_file"]
    if checksum(snapshot) != manifest["snapshot_sha256"]:
        raise ValueError("Snapshot checksum differs from the manifest")
    raw = snapshot.with_suffix("") if snapshot.suffix == ".gz" else snapshot
    if snapshot.suffix == ".gz" and not raw.exists():
        partial = raw.with_suffix(raw.suffix + ".partial")
        if shutil.disk_usage(folder).free < manifest["raw_snapshot_bytes"] + 1024**3:
            raise RuntimeError("Not enough disk space to unpack the snapshot")
        with gzip.open(snapshot, "rb") as source, partial.open("wb") as destination:
            shutil.copyfileobj(source, destination, length=8 * 1024**2)
        if checksum(partial) != manifest["raw_snapshot_sha256"]:
            raise ValueError("Unpacked snapshot checksum mismatch")
        partial.replace(raw)
    if checksum(raw) != manifest["raw_snapshot_sha256"]:
        raise ValueError("Raw snapshot checksum mismatch")
    return manifest, raw


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--folder", type=Path, default=Path.cwd())
    parser.add_argument("--verify-url", help="Verify an already restored server (does not restore or unpack)")
    args = parser.parse_args()
    folder = args.folder.resolve()
    if args.verify_url:
        print(json.dumps(verify(folder, args.verify_url), ensure_ascii=False, indent=2))
        return
    manifest, raw = unpack(folder)
    name, version = manifest["collection"], manifest["qdrant_version"]
    if not manifest["complete"]:
        print("RECOVERY SNAPSHOT: partial ingestion/indexing; use resume_from in the notebook to finish.")
    print(f"Verified: {raw}; {manifest['points']:,} points. Keep the snapshot and restored storage on disk during recovery.")
    print("Run these commands with Docker and curl installed. Use an empty qdrant_storage directory.")
    print("docker run -d --name sbtc-qdrant -p 127.0.0.1:6333:6333 "
          "-e QDRANT__STORAGE__LOW_MEMORY_MODE=no_populate "
          "-e QDRANT__STORAGE__SNAPSHOTS_PATH=/qdrant/storage/snapshots "
          f"-v {shlex.quote(str(folder / 'qdrant_storage') + ':/qdrant/storage')} "
          f"-v {shlex.quote(str(folder) + ':/qdrant/import:ro')} qdrant/qdrant:v{version}")
    print("# Wait until curl --fail http://127.0.0.1:6333/readyz succeeds, then:")
    print("docker exec sbtc-qdrant mkdir -p /qdrant/storage/snapshots")
    print(f"docker exec sbtc-qdrant cp --reflink=auto {shlex.quote('/qdrant/import/' + raw.name)} /qdrant/storage/snapshots/")
    body = json.dumps({"location": f"file:///qdrant/storage/snapshots/{raw.name}", "priority": "snapshot"})
    print(f"curl --fail -X PUT 'http://127.0.0.1:6333/collections/{name}/snapshots/recover?wait=true' "
          f"-H 'Content-Type: application/json' --data {shlex.quote(body)}")
    print(f"curl --fail -X POST http://127.0.0.1:6333/collections/{name}/points/count "
          "-H 'Content-Type: application/json' --data '{\"exact\":true}'")
    print(f"python restore_qdrant_snapshot.py --folder {shlex.quote(str(folder))} --verify-url http://127.0.0.1:6333")


if __name__ == "__main__":
    main()
'''

client_source = r'''"""Read-only dense queries for the restored Qwen3 collection."""

from __future__ import annotations

import json
import math
import os
from collections.abc import Sequence
from pathlib import Path
from urllib.parse import quote

import requests


class QdrantManager:
    """Read-only dense queries; never create collections, indexes or models."""

    def __init__(
        self, url: str | None = None, collection: str | None = None, *,
        dense_vector_name: str | None = None, api_key: str | None = None,
        timeout: float = 30, session: requests.Session | None = None,
    ) -> None:
        self.url = (url or os.getenv("QDRANT_URL", "http://127.0.0.1:6333")).rstrip("/")
        self.collection = collection or os.getenv("QDRANT_COLLECTION", "sbtc_qwen3_embedding_0_6b_1024_dense")
        self.dense_vector_name = dense_vector_name or os.getenv("QDRANT_DENSE_VECTOR_NAME")
        self.api_key = api_key or os.getenv("QDRANT_API_KEY")
        self.timeout = timeout
        self.session = session if session is not None else requests.Session()
        self._owns_session = session is None
        self._config = None
        self._path = f"/collections/{quote(self.collection, safe='')}"

    @classmethod
    def from_manifest(cls, path: str | Path, *, url: str | None = None, **kwargs):
        """Use the exact dense vector name exported by the notebook."""
        manifest = json.loads(Path(path).read_text(encoding="utf-8"))
        if not manifest["complete"]:
            raise ValueError("Recovery snapshot is incomplete; resume the notebook before production queries")
        return cls(url, manifest["collection"], dense_vector_name=manifest["vector_schema"]["dense"], **kwargs)

    def _request(self, method: str, path: str, body: dict | None = None):
        response = self.session.request(
            method, self.url + path, json=body, timeout=self.timeout,
            headers={"api-key": self.api_key} if self.api_key else {},
        )
        response.raise_for_status()
        return response.json()["result"]


    def refresh_collection(self) -> dict:
        """Reload schema after an administrator adds or changes vector indexes."""
        info = self._request("GET", self._path)
        self._config = info["config"]["params"]
        return info


    def _collection_config(self) -> dict:
        if self._config is None:
            self.refresh_collection()
        return self._config


    def _dense_query(self, vector: Sequence[float]) -> dict:
        vectors = self._collection_config()["vectors"]
        if self.dense_vector_name:
            if self.dense_vector_name not in vectors:
                raise ValueError(f"Dense vector {self.dense_vector_name!r} is not configured")
            specification = vectors[self.dense_vector_name]
        else:
            if "size" in vectors:
                specification = vectors
            elif "dense" in vectors:
                self.dense_vector_name = "dense"
                specification = vectors["dense"]
            else:
                raise ValueError("Collection uses named vectors; set dense_vector_name")
        values = [float(value) for value in vector]
        if len(values) != specification["size"]:
            raise ValueError(f"Expected {specification['size']} dense dimensions, got {len(values)}")
        if not all(math.isfinite(value) for value in values) or not any(values):
            raise ValueError("Dense vector must contain finite values and be nonzero")
        query = {"query": values}
        if self.dense_vector_name:
            query["using"] = self.dense_vector_name
        return query


    def search_dense(
        self, dense_vector: Sequence[float], top_k: int = 10, *,
        query_filter: dict | None = None, with_payload: bool | list[str] = True,
    ) -> list[dict]:
        """Search the existing dense vector with cosine similarity."""
        if top_k < 1:
            raise ValueError("top_k must be positive")
        body = {
            **self._dense_query(dense_vector), "limit": top_k,
            "with_payload": with_payload, "with_vector": False,
        }
        if query_filter is not None:
            body["filter"] = query_filter
        return self._request("POST", self._path + "/points/query", body)["points"]


    def close(self) -> None:
        if self._owns_session:
            self.session.close()


    def __enter__(self):
        return self


    def __exit__(self, exc_type, exc_value, traceback):
        self.close()
'''

runner_path.write_text(runner_source, encoding='utf-8')
(output / 'restore_qdrant_snapshot.py').write_text(restore_source, encoding='utf-8')
(output / 'qdrant_db.py').write_text(client_source, encoding='utf-8')


In [ ]:
restore_readme = '# Restore Qwen3 dense snapshot\n\nTải toàn bộ `qdrant_dense_export`. Snapshot chỉ chứa vector `dense` float16,\nHNSW, keyword indexes `chunk_id`/`parent_id`, `chunk_text` và metadata.\nKhông cần corpus JSONL để trả text sau restore; không có BM25/sparse index.\n\nChỉ coi run hoàn tất khi pipeline trả mã 0, `pipeline_status.json` ghi success,\nmanifest có `complete=true` và `restore_verification.verified=true`. Runner đã\nrestore thật snapshot sang collection tạm và kiểm tra exact count, dense\nschema/search, keyword indexes và ánh xạ metadata.\n\n## Restore\n\nDùng Qdrant **1.19.2**, storage trống và giữ database cũ an toàn. Helper mặc định\nchỉ in lệnh Docker/curl để kiểm tra, không tự chạy Docker hoặc sửa database:\n\n```bash\npython restore_qdrant_snapshot.py --folder /absolute/path/qdrant_dense_export\n```\n\nChạy các lệnh được in theo thứ tự: tạo container, đợi `/readyz`, staging snapshot,\nrecover và verify. Nếu cổng 6333/tên container đã dùng, chọn cổng/tên khác.\nCần đủ dung lượng cho file nén, snapshot giải nén, staging và collection restore.\nKhông cần GPU để restore hoặc chạy Qdrant.\n\n```bash\npython restore_qdrant_snapshot.py --folder /absolute/path/qdrant_dense_export \\\n  --verify-url http://127.0.0.1:6333\n```\n\nVerify phải trả `dense: true` và `payload_lookup: true`; không tải model,\nkhông tạo index mới. File nén và snapshot gốc đều được kiểm tra checksum.\n\n## Python client\n\nCần `requests`. Dùng file `qdrant_db.py` đi kèm và chạy từ thư mục export:\n\n```python\nimport json\nfrom qdrant_db import QdrantManager\n\nexample = json.load(open("dense_query_example.json", encoding="utf-8"))\nwith QdrantManager.from_manifest("snapshot_manifest.json") as db:\n    hits = db.search_dense(example["dense_vector"], top_k=10)\n    for hit in hits:\n        payload = hit["payload"]\n        print(hit["score"], payload["chunk_id"], payload["parent_id"], payload["chunk_text"])\n```\n\nĐây là smoke test dùng vector chunk đã lưu, không đánh giá chất lượng retrieval.\nVới câu hỏi mới, encode một dense query vector mới bằng cùng cấu hình:\n\n- Model: `Qwen/Qwen3-Embedding-0.6B`.\n- Revision: `97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3`.\n- Model inference FP16, last-token pooling với left padding, 1024 chiều.\n- Gộp cửa sổ và chuẩn hóa L2 bằng float32; Qdrant lưu float16, Cosine.\n- Query: `Instruct: Given a web search query, retrieve relevant passages that answer the query\\nQuery: {query}`\n  (`\\n` là ký tự xuống dòng). Document không thêm instruction.\n- Query dài dùng cùng cửa sổ 8192 token và content-length-weighted mean rồi L2.\n  Input không có token dùng một EOS token.\n\nGọi `db.search_dense(dense_query, top_k=10)`. Client không tự tải model/encode\nquery, không tạo collection/index hoặc tính sparse embedding. Nếu máy khác,\ndùng SSH tunnel và truyền `url` vào `from_manifest()`.\n\n## Lỗi, quota và resume\n\nTên `RECOVERY_...` hoặc `complete=false` là snapshot chưa hoàn tất. Gắn Output\nlàm Input, đặt `resume_from`, dùng state/output mới để tiếp tục. Chỉ dùng lại\nsnapshot dense với cùng source, model/revision, dimension và datatype; không\nresume snapshot hybrid. Các chunk rỗng hoặc không eligible vẫn được giữ.\n\nOutput có giới hạn 18 GiB dự phòng. Nếu snapshot vượt ngân sách ngay cả sau\nnén, runner báo lỗi và giữ snapshot trong state của runtime hiện tại; dữ liệu\n`/tmp` sẽ mất khi runtime kết thúc. Không coi lỗi export là kết quả đã lưu.\n\nTài liệu: https://qdrant.tech/documentation/operations/snapshots/\n'
(output / 'README_RESTORE.md').write_text(restore_readme, encoding='utf-8')


## 4. Chạy toàn bộ

Xác minh source hash/count và ID cha–con → kiểm tra dung lượng → tạo dense collection và keyword indexes → hai GPU encode/upsert → exact count → bật HNSW, đợi optimizer → thử dense search và lookup metadata → tạo snapshot → restore thử và xác minh schema/count/search/metadata → checksum/nén/export và manifest.

Nếu worker lỗi, dừng worker còn lại trước khi xuất RECOVERY. Replay dùng cùng UUID và payload nên không tạo điểm trùng. Chỉ coi hoàn tất khi return code bằng 0, `complete=true` và `restore_verification.verified=true`. State trong `/tmp` chỉ tồn tại trong runtime hiện tại.


In [ ]:
result = subprocess.run([sys.executable, '-u', str(runner_path), 'run', '--config', str(config_path)])
pipeline_ok = result.returncode == 0
print('Pipeline return code:', result.returncode)
if not pipeline_ok:
    print('PIPELINE FAILED: inspect logs and pipeline_status.json; any RECOVERY snapshot is incomplete.')

## 5. Restore và dense search

Tải snapshot, `snapshot_manifest.json`, `restore_qdrant_snapshot.py`, `qdrant_db.py`, `dense_query_example.json`, `README_RESTORE.md` cùng log/status. Dùng Qdrant **1.19.2** và storage trống. Helper kiểm tra checksum rồi in lệnh Docker/curl; `--verify-url` kiểm tra exact count, dense schema/search và metadata sau restore.

Client `QdrantManager.from_manifest(...)` đọc đúng collection/vector name; gọi `search_dense(...)` với vector query mới. Ví dụ đi kèm dùng stored vector để smoke test. Snapshot không chứa weights Qwen3, không có sparse vectors. Snapshot dành cho single-node, không chứa aliases.


In [ ]:
from IPython.display import FileLink, display

manifest_path = output / 'snapshot_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    complete = pipeline_ok and manifest['complete'] and manifest.get('restore_verification', {}).get('verified', False)
    print('SUCCESS' if complete else 'RECOVERY ONLY / INCOMPLETE')
    print(f"{manifest['points']:,} points; {manifest['snapshot_bytes'] / 1024**3:.2f} GiB; Qdrant {manifest['qdrant_version']}")
    for filename in (manifest['snapshot_file'], 'snapshot_manifest.json', 'restore_qdrant_snapshot.py', 'qdrant_db.py', 'dense_query_example.json', 'README_RESTORE.md'):
        if (output / filename).exists():
            display(FileLink(str(output / filename)))
else:
    print('NO SNAPSHOT EXPORTED. Inspect notebook stderr, qdrant.log and pipeline_status.json.')
